# Cascade JEPA with LoRA on Qwen3-1.7B, NOUL questions

The frozen run (`noul_cascade_decoder.ipynb`) gave the cascade +3–4 points over the same heads without JEPA. Here the decoder is also fine-tuned with **LoRA**, identically in every arm, so the arms still differ only in JEPA.

| Arm | LoRA | Stage 1 (Q·S) | Decision read-out | Compared with |
|---|---|---|---|---|
| **A** cascade | yes | predictor on layer `layer` → ẑ₁ ≈ Q·S | ẑ₁ → ẑ₂, cosine to the yes/no vectors / τ | – |
| **B** same heads, no JEPA | yes | – | the same predictor → 2 logits | **A − B: JEPA** |
| **L** standard LoRA classifier | yes | – | the model's own next-token logits, {Yes, yes} vs {No, no} | **A − L: JEPA vs the usual way** |

- **Q·S** is made once by the **frozen base** model (before any LoRA), with the recipe that passed the check on this model: standardised space, attention ×16, minus the state's plain average. So the target does not move while LoRA trains.
- **Matched training:** `stage1_epochs` + `stage2_epochs` epochs in every arm; A trains stage 1 alone first, then the decision with stage 1 kept on; B and L train the decision throughout. Warm-up, linear decay; the best decision epoch by validation KL is kept.
- **Checkpoints** after every epoch (LoRA weights, heads, optimiser), with resume. A speed check after `speed_check_steps` steps prints the projected time and cost.

Measures: validation, our test split, and the 600 Typed Decisions NOUL questions (Signpost-D-4B: 0.783 accuracy).

## Settings

In [ ]:
import os


def setting(name, default):
    value = os.environ.get(name.upper())
    if value is None:
        return default
    if isinstance(default, list):
        kind = type(default[0]) if default else str
        return [kind(v.strip()) for v in value.split(",") if v.strip()]
    if isinstance(default, bool):
        return value.strip().lower() in ("1", "true", "yes")
    return type(default)(value)


CONFIG = {
    "arm": setting("arm", "A"),
    "seed": setting("seed", 42),
    "model_name": setting("model_name", "Qwen/Qwen3-1.7B"),
    "layer": setting("layer", 20),
    "data_path": setting("data_path", "data/signpost_d_decisions/train.jsonl"),
    "data_url": setting("data_url", "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/"
                                    "decider-jepa/data/signpost_d_decisions/train.jsonl"),
    "max_sequence_length": setting("max_sequence_length", 1536),
    "split_seed": setting("split_seed", 42),
    "train_ratio": setting("train_ratio", 0.80),
    "validation_ratio": setting("validation_ratio", 0.10),
    "encode_batch": setting("encode_batch", 8),

    # Q·S (frozen base): the recipe that passed on this model
    "qs_space": setting("qs_space", "standardised"),
    "qs_sharpness": setting("qs_sharpness", 16.0),
    "qs_residual": setting("qs_residual", True),
    "qs_gate_same_state_cos": setting("qs_gate_same_state_cos", 0.95),

    "target_words": setting("target_words", ["yes", "no"]),
    "target_repeats": setting("target_repeats", 64),
    "temperature": setting("temperature", 0.1),

    # LoRA
    "lora_r": setting("lora_r", 16),
    "lora_alpha": setting("lora_alpha", 32),
    "lora_dropout": setting("lora_dropout", 0.05),
    "lora_targets": setting("lora_targets", ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]),

    # heads
    "pred_width": setting("pred_width", 384),
    "pred_layers": setting("pred_layers", 2),
    "pred_heads": setting("pred_heads", 6),
    "dropout": setting("dropout", 0.1),

    # training
    "batch_size": setting("batch_size", 8),
    "eval_batch": setting("eval_batch", 16),
    "lora_lr": setting("lora_lr", 1e-4),
    "head_lr": setting("head_lr", 3e-4),
    "weight_decay": setting("weight_decay", 0.01),
    "warmup_fraction": setting("warmup_fraction", 0.1),
    "stage1_epochs": setting("stage1_epochs", 3),
    "stage2_epochs": setting("stage2_epochs", 5),
    "stage1_weight_in_stage2": setting("stage1_weight_in_stage2", 1.0),
    "grad_clip": setting("grad_clip", 1.0),
    "speed_check_steps": setting("speed_check_steps", 50),

    "td_dataset": setting("td_dataset", "LocalLLaMA/typed-decisions"),
    "td_file": setting("td_file", "all/test-00000-of-00001.parquet"),
    "out_dir": setting("out_dir", "cascade_lora"),
}

ARMS = {"A": {"readout": "cosine", "stage1": True},
        "B": {"readout": "logits", "stage1": False},
        "L": {"readout": "lm", "stage1": False}}
ARM = ARMS[CONFIG["arm"]]

for k, v in CONFIG.items():
    print(f"{k:26s}: {v}")
print("arm:", CONFIG["arm"], ARM)

## Data (as in the other NOUL notebooks)

In [ ]:
import json
import random
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
device = torch.device("cuda")

DATA_PATH = Path(CONFIG["data_path"])
if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(CONFIG["data_url"], DATA_PATH)


def parsed(value):
    return json.loads(value) if isinstance(value, str) else value


def noul_samples(records):
    out = []
    for record in records:
        state_text = "State:\n" + str(record["state"])
        for qid, q in record["questions"].items():
            if str(q.get("type", "")).lower() != "noul":
                continue
            gold = {str(k): float(v) for k, v in record["gold"][qid]["probabilities"].items()}
            question_text = "Question:\n" + str(q.get("instructions"))
            out.append({"group": record["id"], "question_id": qid,
                        "state_text": state_text, "question_text": question_text,
                        "gold": [gold["true"], gold["false"]]})
    return out


records = [json.loads(line) for line in DATA_PATH.open(encoding="utf-8") if line.strip()]
samples = noul_samples(records)

groups = pd.Series([s["group"] for s in samples]).drop_duplicates().to_numpy()
shuffled = np.random.default_rng(CONFIG["split_seed"]).permutation(groups)
n_train = int(len(groups) * CONFIG["train_ratio"])
n_val = int(len(groups) * CONFIG["validation_ratio"])
split_of = {g: "train" for g in shuffled[:n_train]}
split_of.update({g: "validation" for g in shuffled[n_train:n_train + n_val]})
split_of.update({g: "test" for g in shuffled[n_train + n_val:]})
splits = {name: [s for s in samples if split_of[s["group"]] == name] for name in ("train", "validation", "test")}

td_frame = pd.read_parquet(hf_hub_download(CONFIG["td_dataset"], CONFIG["td_file"], repo_type="dataset"))
td_records = [{"id": r["id"], "state": parsed(r["state"]), "questions": parsed(r["questions"]),
               "gold": parsed(r["gold"])} for r in td_frame.to_dict("records")]
splits["td_test"] = noul_samples(td_records)

GOLD = {name: torch.tensor([r["gold"] for r in rows]) for name, rows in splits.items()}
print({k: len(v) for k, v in splits.items()})

## The model, the prompt, and the frozen-base targets

The base model is loaded once. **Before LoRA is attached**, it makes (for arms A and B) the per-dimension statistics of layer `layer` over the training prompts (to standardise the heads' input), and (for A) Q·S and the yes/no vectors.

In [ ]:
import math
import time

import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(CONFIG["model_name"], dtype=torch.bfloat16).to(device)
HIDDEN = model.config.hidden_size
LAYER = CONFIG["layer"]


def prompt_of(row):
    messages = [{"role": "user", "content": row["state_text"] + "\n\n" + row["question_text"] + "\n\nAnswer yes or no."}]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, enable_thinking=False)


def single_token(text):
    ids = tokenizer.encode(text, add_special_tokens=False)
    assert len(ids) == 1, (text, ids)
    return ids[0]


YES_IDS = [single_token(t) for t in ("Yes", "yes")]
NO_IDS = [single_token(t) for t in ("No", "no")]


def encode(texts):
    return tokenizer(texts, padding=True, truncation=True, add_special_tokens=False,
                     max_length=CONFIG["max_sequence_length"], return_tensors="pt").to(device)


@torch.no_grad()
def frozen_states(texts, how):
    """how: 'tokens' -> [n, hidden] per text (bf16); 'mean' -> [hidden] per text."""
    model.eval()
    out = [None] * len(texts)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    for s in range(0, len(order), CONFIG["encode_batch"]):
        idx = order[s:s + CONFIG["encode_batch"]]
        batch = encode([texts[i] for i in idx])
        hs = model.model(**batch, output_hidden_states=True).hidden_states[LAYER]
        lengths = batch["attention_mask"].sum(1)
        for r, i in enumerate(idx):
            t = hs[r, :int(lengths[r])]
            out[i] = t.clone() if how == "tokens" else t.float().mean(0)
    return out


def running_stats(tensors):
    total = torch.zeros(HIDDEN, dtype=torch.float64, device=device)
    squares = torch.zeros(HIDDEN, dtype=torch.float64, device=device)
    count = 0
    for t in tensors:
        x = t.double()
        total += x.sum(0)
        squares += (x * x).sum(0)
        count += len(x)
    mean = total / count
    return mean.float(), (squares / count - mean * mean).clamp_min(1e-12).sqrt().float().clamp_min(1e-6)


t0 = time.perf_counter()
if ARM["readout"] != "lm":
    # input statistics for the heads: layer-`LAYER` states of the training prompts, frozen base
    train_prompts = [prompt_of(r) for r in splits["train"]]
    stats_sum = []
    for s in range(0, len(train_prompts), 256):
        stats_sum += frozen_states(train_prompts[s:s + 256], "tokens")
        if len(stats_sum) > 1024:
            break                                     # 1,024 prompts are plenty for the statistics
    IN_MEAN, IN_STD = running_stats(stats_sum)
    del stats_sum
    print(f"head input statistics from 1,024 training prompts | {time.perf_counter() - t0:.0f} s")

if ARM["stage1"]:
    Q = {name: torch.stack(frozen_states([r["question_text"] for r in rows], "mean")) for name, rows in splits.items()}
    S = {}
    for name, rows in splits.items():
        unique = sorted({r["state_text"] for r in rows})
        by_text = dict(zip(unique, frozen_states(unique, "tokens")))
        S[name] = [by_text[r["state_text"]] for r in rows]
    TOKEN_MEAN, TOKEN_STD = running_stats({id(t): t for t in S["train"]}.values())

    def qs_target(q, Sx):
        Sx, q = Sx.float(), q.float()
        if CONFIG["qs_space"] == "standardised":
            Sx, q = (Sx - TOKEN_MEAN) / TOKEN_STD, (q - TOKEN_MEAN) / TOKEN_STD
        w = torch.softmax(CONFIG["qs_sharpness"] * (Sx @ q) / math.sqrt(Sx.shape[1]), dim=0)
        t = w @ Sx
        return t - Sx.mean(0) if CONFIG["qs_residual"] else t

    QS = {name: torch.stack([qs_target(Q[name][i], S[name][i]) for i in range(len(S[name]))]) for name in splits}

    by_group = {}
    for i, r in enumerate(splits["train"]):
        by_group.setdefault(r["group"], []).append(i)
    same = [F.cosine_similarity(QS["train"][a], QS["train"][b], dim=0).item()
            for idx in by_group.values() for j, a in enumerate(idx) for b in idx[j + 1:]]
    print(f"Q·S check: same state {np.mean(same):.4f} (gate < {CONFIG['qs_gate_same_state_cos']})")
    assert np.mean(same) < CONFIG["qs_gate_same_state_cos"]

    CHOICES = torch.stack(frozen_states([" ".join([w] * CONFIG["target_repeats"]) for w in CONFIG["target_words"]],
                                        "mean")).float()
    c = F.cosine_similarity(CHOICES[0], CHOICES[1], dim=0).item()
    ceiling = 1 / (1 + math.exp(-2 * math.sin(math.acos(max(-1.0, min(1.0, c))) / 2) / CONFIG["temperature"]))
    print(f"choice vectors: cos {c:.4f}, highest gold probability {ceiling:.4f}")
    del Q, S
    torch.cuda.empty_cache()
print(f"frozen-base pass done in {time.perf_counter() - t0:.0f} s")

## LoRA and the heads

LoRA on the attention and MLP projections; gradient checkpointing. The heads (arms A and B) read layer `layer`'s token states, standardised with the frozen-base statistics: a learned query token and `pred_layers` transformer layers → ẑ₁; then a cosine head (A) or 2 logits (B). Arm L reads the language-model head at the last prompt token: log-sum-exp of {Yes, yes} and {No, no}.

In [ ]:
import copy

import torch.nn as nn
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict

torch.manual_seed(CONFIG["seed"])
model = get_peft_model(model, LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"],
                                         lora_dropout=CONFIG["lora_dropout"],
                                         target_modules=CONFIG["lora_targets"], task_type="CAUSAL_LM"))
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False
model.print_trainable_parameters()

LM_HEAD = model.get_output_embeddings().weight          # frozen
LABEL_ROWS = torch.tensor(YES_IDS + NO_IDS, device=device)


class Heads(nn.Module):
    def __init__(self, readout):
        super().__init__()
        w = CONFIG["pred_width"]
        self.readout = readout
        self.inp = nn.Linear(HIDDEN, w)
        self.query = nn.Parameter(torch.randn(1, 1, w) * 0.02)
        layer = nn.TransformerEncoderLayer(w, CONFIG["pred_heads"], 4 * w, CONFIG["dropout"],
                                           batch_first=True, norm_first=True, activation="gelu")
        self.layers = nn.TransformerEncoder(layer, CONFIG["pred_layers"])
        self.z1 = nn.Sequential(nn.LayerNorm(w), nn.Linear(w, HIDDEN))
        out = HIDDEN if readout == "cosine" else 2
        self.head = nn.Sequential(nn.LayerNorm(HIDDEN), nn.Linear(HIDDEN, HIDDEN), nn.GELU(), nn.Linear(HIDDEN, out))

    def forward(self, tokens, pad):
        x = self.inp((tokens.float() - IN_MEAN) / IN_STD)
        x = torch.cat([self.query.expand(len(x), -1, -1), x], dim=1)
        pad = torch.cat([torch.zeros(len(x), 1, dtype=torch.bool, device=x.device), pad], dim=1)
        z1 = self.z1(self.layers(x, src_key_padding_mask=pad)[:, 0])
        h = self.head(z1)
        if self.readout == "cosine":
            return z1, F.cosine_similarity(h.unsqueeze(1), CHOICES.to(device).unsqueeze(0), dim=-1) / CONFIG["temperature"]
        return z1, h


heads = Heads(ARM["readout"]).to(device) if ARM["readout"] != "lm" else None


def forward(name, idx):
    """-> (ẑ₁ or None, 2 logits [yes, no])"""
    batch = encode([prompt_of(splits[name][i]) for i in idx])
    out = model.base_model.model.model(**batch, output_hidden_states=True)      # the decoder stack, with LoRA
    pad = batch["attention_mask"] == 0
    if ARM["readout"] == "lm":
        last = batch["attention_mask"].sum(1) - 1
        final = out.last_hidden_state[torch.arange(len(idx), device=device), last]
        label = final.float() @ LM_HEAD[LABEL_ROWS].float().T                 # [B, 4]
        logits = torch.stack([torch.logsumexp(label[:, :2], 1), torch.logsumexp(label[:, 2:], 1)], dim=1)
        return None, logits
    return heads(out.hidden_states[LAYER], pad)


def kl(logits, gold):
    return (gold * (gold.clamp_min(1e-12).log() - F.log_softmax(logits.float(), dim=1))).sum(1)


@torch.no_grad()
def evaluate(name):
    model.eval()
    if heads is not None:
        heads.eval()
    n = len(splits[name])
    order = sorted(range(n), key=lambda i: len(splits[name][i]["state_text"]))
    logits_all, cos_all = [], []
    for s in range(0, n, CONFIG["eval_batch"]):
        idx = order[s:s + CONFIG["eval_batch"]]
        z1, logits = forward(name, idx)
        logits_all.append(logits.float())
        if z1 is not None and ARM["stage1"]:
            cos_all.append(F.cosine_similarity(z1, QS[name][idx], dim=1))
    logits = torch.cat(logits_all)
    gold = GOLD[name][order].to(device)
    p = F.softmax(logits, dim=1)
    return {"stage1_cos": torch.cat(cos_all).mean().item() if cos_all else float("nan"),
            "kl": kl(logits, gold).mean().item(), "uniform_kl": kl(torch.zeros_like(gold), gold).mean().item(),
            "top_choice": (p.argmax(1) == gold.argmax(1)).float().mean().item(),
            "brier": ((p - gold) ** 2).sum(1).mean().item(), "p_true_std": p[:, 0].std().item()}

## Training, with a checkpoint after every epoch

In [ ]:
OUT_DIR = Path(CONFIG["out_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT = OUT_DIR / f"arm_{CONFIG['arm']}_latest.pt"
BEST = OUT_DIR / f"arm_{CONFIG['arm']}_best.pt"

lora_params = [p for p in model.parameters() if p.requires_grad]
param_groups = [{"params": lora_params, "lr": CONFIG["lora_lr"]}]
if heads is not None:
    param_groups.append({"params": list(heads.parameters()), "lr": CONFIG["head_lr"]})
opt = torch.optim.AdamW(param_groups, weight_decay=CONFIG["weight_decay"])

n_train = len(splits["train"])
steps_per_epoch = math.ceil(n_train / CONFIG["batch_size"])
total_epochs = CONFIG["stage1_epochs"] + CONFIG["stage2_epochs"]
total_steps = total_epochs * steps_per_epoch
warmup = int(CONFIG["warmup_fraction"] * total_steps)


def lr_factor(step):
    if step < warmup:
        return (step + 1) / max(1, warmup)
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup))


sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)


def trained_state():
    return {"lora": {k: v.detach().cpu() for k, v in get_peft_model_state_dict(model).items()},
            "heads": {k: v.detach().cpu() for k, v in heads.state_dict().items()} if heads is not None else None}


def load_trained(saved):
    set_peft_model_state_dict(model, saved["lora"])
    if heads is not None:
        heads.load_state_dict(saved["heads"])


def save(path, obj):
    tmp = path.with_suffix(".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


state = {"epoch": 0, "best_kl": float("inf"), "best_epoch": 0, "history": []}
if CKPT.exists():
    saved = torch.load(CKPT, map_location="cpu", weights_only=False)
    load_trained(saved["trained"]); opt.load_state_dict(saved["opt"]); sched.load_state_dict(saved["sched"])
    state = saved["state"]
    print(f"RESUMED after epoch {state['epoch']}")

RATES = {"L4": 0.80, "A10": 1.10, "L40S": 1.95, "A100": 2.50, "H100": 3.95}
gpu = torch.cuda.get_device_name(0)
rate = next((v for k, v in RATES.items() if k in gpu), None)
speed_checked = False

for epoch in range(state["epoch"] + 1, total_epochs + 1):
    in_stage1 = ARM["stage1"] and epoch <= CONFIG["stage1_epochs"]
    w1 = (1.0 if in_stage1 else CONFIG["stage1_weight_in_stage2"]) if ARM["stage1"] else 0.0
    w_kl = 0.0 if in_stage1 else 1.0
    model.train()
    if heads is not None:
        heads.train()
    perm = torch.randperm(n_train, generator=torch.Generator().manual_seed(CONFIG["seed"] + epoch)).tolist()
    total, t0 = 0.0, time.perf_counter()
    for step, s in enumerate(range(0, n_train, CONFIG["batch_size"]), start=1):
        idx = perm[s:s + CONFIG["batch_size"]]
        z1, logits = forward("train", idx)
        loss = w_kl * kl(logits, GOLD["train"][idx].to(device)).mean()
        if w1:
            loss = loss + w1 * (1 - F.cosine_similarity(z1, QS["train"][idx], dim=1)).mean()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for g in param_groups for p in g["params"]], CONFIG["grad_clip"])
        opt.step(); sched.step()
        total += loss.item() * len(idx)
        if not speed_checked and step == CONFIG["speed_check_steps"]:
            speed_checked = True
            per_step = (time.perf_counter() - t0) / step
            left_h = per_step * (total_steps - (epoch - 1) * steps_per_epoch - step) / 3600
            cost = f", about ${left_h * rate:.2f} at ${rate}/h" if rate else ""
            print(f"SPEED CHECK on {gpu}: {per_step:.2f} s/step -> {left_h * 60:.0f} min of training left{cost}")
    v = evaluate("validation")
    stage = "stage 1" if in_stage1 else "decision"
    print(f"arm {CONFIG['arm']} epoch {epoch} ({stage}) | train loss {total / n_train:.4f} | "
          f"{(time.perf_counter() - t0) / 60:.1f} min | val stage-1 cos {v['stage1_cos']:.4f} | "
          f"val KL {v['kl']:.4f} (uniform {v['uniform_kl']:.4f}) | top {v['top_choice']:.3f} | "
          f"P(true) std {v['p_true_std']:.3f}")
    state["history"].append({"epoch": epoch, "stage": stage, "train_loss": total / n_train, **v})
    if not in_stage1 and v["kl"] < state["best_kl"]:
        state["best_kl"], state["best_epoch"] = v["kl"], epoch
        save(BEST, trained_state())
    state["epoch"] = epoch
    save(CKPT, {"trained": trained_state(), "opt": opt.state_dict(), "sched": sched.state_dict(), "state": state})

print(f"best decision epoch {state['best_epoch']} (validation KL {state['best_kl']:.4f})")

## Results (best decision epoch)

In [ ]:
load_trained(torch.load(BEST, map_location="cpu", weights_only=False))
results = {name: evaluate(name) for name in ("train", "validation", "test", "td_test")}

table = pd.DataFrame(results).T
table.insert(0, "arm", CONFIG["arm"])
pd.set_option("display.width", 250)
display(table.round(4))

td = results["td_test"]
print(f"\nARM {CONFIG['arm']} RESULT (best epoch {state['best_epoch']}) | validation top "
      f"{results['validation']['top_choice']:.4f} KL {results['validation']['kl']:.4f} | test top "
      f"{results['test']['top_choice']:.4f} KL {results['test']['kl']:.4f} | Typed Decisions NOUL accuracy "
      f"{td['top_choice']:.4f} KL {td['kl']:.4f} (uniform {td['uniform_kl']:.4f}; Signpost-D-4B 0.7833 / 0.0952)")

with open(OUT_DIR / f"arm_{CONFIG['arm']}_results.json", "w") as f:
    json.dump({"config": CONFIG, "arm": ARM, "best_epoch": state["best_epoch"], "results": results,
               "history": state["history"]}, f, indent=1)
CKPT.unlink(missing_ok=True)
print("saved", OUT_DIR)